## nb_00_dbutils

Contains common modules used for transformation

#### function list
- load_table
- fill_null_columns
- remove_duplicates
- drop_no_null_columns
- validate_no_nulls
- validate_column_values
- validate_primary_keys
- validate_foreign_keys
- write_table

### Imports

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, count, when, lit, min, max,
    substring, concat, to_timestamp, to_date,
)

StatementMeta(, d884e28d-b3f5-481e-8e63-db4e5d5a13aa, 3, Finished, Available, Finished, False)

### Function: Load table
This is a generic function that can used to load any type of table, bronze, silver, gold, etc

In [2]:
def load_table(spark, table_name: str, columns: list[str] | None = None) -> DataFrame:
    """
    Read bronze table, if columns are not specified, 
    select all columns in the returned DataFrame.
    """

    query = f"""
        SELECT *
        FROM {table_name}
    """
    df = spark.sql(query)
    print(f"✅ Loaded {table_name}: {df.count()} rows")

    if columns is None:
       return df
    
    return df.select(*columns)


StatementMeta(, d884e28d-b3f5-481e-8e63-db4e5d5a13aa, 4, Finished, Available, Finished, False)

## Function: fill null columns

In [3]:
def fill_null_columns(df: DataFrame, keys: dict[str, str]) -> DataFrame:
    """Fill null values in specified columns with the given values."""

    for c, value in keys.items():
        before = df.filter(col(c).isNull()).count()
        filled_cols = df.fillna(value, subset=[c])
        after = filled_cols.filter(col(c).isNull()).count()
        print(f"🔁 Filled {before - after} null row(s) in '{c}' with '{value}'")

    return filled_cols


StatementMeta(, d884e28d-b3f5-481e-8e63-db4e5d5a13aa, 5, Finished, Available, Finished, False)

### Function: remove duplicates

In [4]:
def remove_duplicates(df: DataFrame, columns: list[str]) -> DataFrame:
    """Drop duplicate rows based on the given key columns, keeping one row per key."""

    before = df.count()
    deduped = df.dropDuplicates(columns)
    after = deduped.count()
    print(f"🔁 Removed {before - after} duplicate row(s) based on {columns}")

    return deduped

StatementMeta(, d884e28d-b3f5-481e-8e63-db4e5d5a13aa, 6, Finished, Available, Finished, False)

### Function: drop rows with nulls in required columns

In [5]:
def drop_no_null_columns(df: DataFrame, columns: list[str]) -> DataFrame:
    """Report, then drop, rows that have a null in any of the given columns."""

    # create a callable to count for null values
    null_counts = df.select([
        count(when(col(c).isNull(), 1)).alias(c) for c in columns
    ]).first()

    for c in columns:
        n = null_counts[c]
        if n > 0:
            print(f"⚠️ '{c}' has {n} null value(s) — those rows will be dropped")

    before = df.count()
    cleaned = df.na.drop(subset=columns)
    after = cleaned.count()
    
    if before != after:
        print(f"🧹 Dropped {before - after} row(s) with nulls in {columns}")

    return cleaned

StatementMeta(, d884e28d-b3f5-481e-8e63-db4e5d5a13aa, 7, Finished, Available, Finished, False)

### Function: validate no nulls in required columns

In [ ]:
def validate_no_nulls(df: DataFrame, columns: list[str]) -> DataFrame:
    """Report, then drop, rows that have a null in any of the given columns."""

    # create a callable to count for null values
    null_counts = df.select([
        count(when(col(c).isNull(), 1)).alias(c) for c in columns
    ]).first()

    for c in columns:
        n = null_counts[c]
        if n > 0:
            raise RuntimeError(f"⚠️ '{c}' has {n} null value(s)")

    return df

### Function: validate column against valid values

In [6]:
def validate_column_values(df: DataFrame, column: str, values: tuple[str]) -> DataFrame:
    """ check column for no unexpect invalid values """
    
    invalid = df.filter(~col(column).isin(*values))
    invalid_count = invalid.count()
    if invalid_count > 0:
        raise RuntimeError(f"⚠️ Found {invalid_count} row(s) with an invalid {column}")

    return df

StatementMeta(, d884e28d-b3f5-481e-8e63-db4e5d5a13aa, 8, Finished, Available, Finished, False)

### Function: validate Primary key(s)

In [ ]:
def validate_primary_keys(df: DataFrame, keys: list[str]) -> DataFrame:
    """Raise if any row's primary key is not unique.

    Intended to run before writing table, as a hard guardrail:
    a violation here means the natural key assumption doesn't hold and the
    pipeline should fail loudly rather than write bad data to silver.
    """
    dupes = (
        df.groupBy(*keys)
        .count()
        .filter(col("count") > 1)
    )
    dupe_count = dupes.count()
    
    if dupe_count > 0:
        print(f"❌ Found {dupe_count} primary key value(s) in {keys} with duplicate rows:")
        dupes.show(truncate=False)
        raise ValueError(
            f"Primary key violation: {dupe_count} duplicate value(s) of {keys} in silver.game"
        )
    print(f"✅ Primary key check passed — {keys} is unique across {df.count()} row(s)")
    return df

### Function: (internal) find foreign key violations

In [ ]:
def _find_fk_violations(df: DataFrame, ftable: DataFrame, keys: list[str]) -> DataFrame:
    """Returns rows in df with no matching foreign key in ftable."""
    
    if len(keys) == 1:
        return df.join(ftable, on=keys[0], how="left_anti")
    elif len(keys) == 2:
        return df.join(ftable, df[keys[0]] == ftable[keys[1]], how="left_anti")
    else:
        raise ValueError(f"❌ Maximum 2 items in parameter keys: {keys}")

### Function: drop rows with foreign key violations

In [ ]:
def drop_foreign_key_violations(df: DataFrame, ftable: DataFrame, keys: list[str]) -> DataFrame:
    """Drops rows in df whose foreign key has no match in ftable. Returns filtered df."""

    violations = _find_fk_violations(df, ftable, keys)   
    count = violations.count()
    
    if count > 0:
        print(f"⚠️ Dropping {count} rows with foreign key violations for {keys}")
        if len(keys) == 1:
            return df.join(ftable, on=keys[0], how="left_semi")
        return df.join(ftable, df[keys[0]] == ftable[keys[1]], how="left_semi")
    
    print(f"✅ Foreign key check passed, no drops applied — {keys}")
    return df

### Function: validate Secondary key(s)

In [ ]:
def validate_foreign_keys(df: DataFrame, ftable: DataFrame, keys: list[str]) -> DataFrame:
    """Raises ValueError if any foreign key violations exist. Returns df unchanged."""

    violations = _find_fk_violations(df, ftable, keys)
    count = violations.count()
    
    if count > 0:
        sample = violations.limit(5).collect()
        raise ValueError(
            f"❌ Foreign key violation for {keys[0]}: {count} rows not found in referenced table. "
            f"Example violations: {sample}"
        )

    print(f"✅ Foreign key check passed — {keys}")
    return df

### Function: write to table

In [7]:
def write_table(df: DataFrame, table_name: str) -> None:
    """ writes to a delta table, overwrite even if exists """

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"✅ Wrote {table_name} ({df.count()} rows, {len(df.columns)} columns)")

StatementMeta(, d884e28d-b3f5-481e-8e63-db4e5d5a13aa, 9, Finished, Available, Finished, False)